> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — chromosome A/D GRMs and LOCO algebra

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=320,n_variants=1000,n_traits=8,seed=SEED,genotype_kwargs={"n_chromosomes":5,"generations":100},trait_kwargs=cfg)
X=sim.genotypes.to_pandas(); Y=sim.phenotypes.to_pandas()
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

rows=[]
for chrom in sorted(vigor.result_table(sim, "variant_info").chrom.astype(str).unique()):
    cols=vigor.result_table(sim, "variant_info").index[vigor.result_table(sim, "variant_info").chrom.astype(str)==chrom]
    g=vigor.genomic_grms_from_genotypes(X.loc[:,cols],include=("additive","dominance"),dtype=np.float64,block_variants=512,blas_threads=2)
    rows.append((chrom,len(cols),float(np.mean(np.diag(g["additive"].to_pandas()))),float(np.mean(np.diag(g["dominance"].to_pandas())))) )
pd.DataFrame(rows,columns=["chromosome","variants","diag_A","diag_D"])


The real-data notebook uses `VigorStore` to persist chromosome GRMs and reconstruct genome-wide/LOCO kernels without re-reading PLINK.

In [ ]:
from tempfile import TemporaryDirectory
from vigor.testutils import write_plink
with TemporaryDirectory() as td:
    prefix=Path(td)/"genotypes"
    vi=vigor.result_table(sim, "variant_info").loc[X.columns]
    write_plink(prefix,X.to_numpy(),X.index.astype(str),vi.chrom.astype(str),vi.pos.to_numpy())
    store=vigor.open_store(Path(td)/"grms.zarr",mode="a",lazy=False)
    vigor.build_grms(prefix,store=store,components=("additive","dominance"),sample_ids=X.index,
        fit_mask=fitmask,block_size=256,blas_threads=2,progress=False,persist_genomewide=True)
    chromosome=str(vi.chrom.iloc[0])
    loco=vigor.open_grm(store,component="additive",loco=chromosome,materialize=True)
    assert loco.shape==(len(Y),len(Y))
    print("Cached LOCO:",chromosome,loco.shape)
